[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session3/student/Assignment_Pharmacy_Pipeline_Student.ipynb)

# Session 3 Assignment: The Complete Pharmacy Intelligence Pipeline
## CCI — Clinical AI Development

**Due:** Before Session 4  
**Estimated effort:** 4-6 hours  
**Submission:** Push to your `cci-course-notebooks` GitHub repo under `assignments/session-3/`

### Clinical Scenario
> KHCC's pharmacy department processes hundreds of medication orders daily through the VistA system. Pharmacists need to verify orders, check for drug interactions, and answer clinical questions about prescribing patterns. Currently this requires manual SQL queries and cross-referencing paper notes. You will build an end-to-end pharmacy intelligence pipeline that loads medication data into SQL, extracts drug interactions from clinical notes using structured AI output, and provides a natural language query interface for pharmacists.

### What You Will Build
This assignment ties together **all five Session 3 labs**:
- **Lab 1** (API Fundamentals) — API calls, token awareness
- **Lab 2** (Structured Output) — Pydantic extraction from clinical notes
- **Lab 3** (Tool Calling) — stretch challenge
- **Lab 4** (CSV to SQL) — loading pharmacy data, writing queries
- **Lab 5** (Text-to-SQL) — natural language pharmacy queries

### Grading
| Part | Weight | Description |
|------|--------|-------------|
| Part 1 | 25% | CSV to SQL — Load and query pharmacy data |
| Part 2 | 25% | Structured Output — Extract drug info from clinical notes |
| Part 3 | 25% | Text-to-SQL — Pharmacist natural language interface |
| Part 4 | 15% | Critical Analysis — Clinical risks and limitations |
| Part 5 | 10% | Stretch Challenge — Tool calling OR benchmarking |

### Anti-Shortcut Rules
- You MUST use the provided data files (`vista_pharmacy.csv`, `vista_patients.csv`, `synthetic_clinical_notes.json`)
- Your Part 4 analysis must reference YOUR specific implementation
- All outputs must reproduce from a top-to-bottom notebook run
- Push to GitHub with at least 3 separate commits with descriptive messages
- No real patient data in the notebook

---
## Setup

In [ ]:
# Setup — clone the repo and install packages
import os

if not os.path.exists('/content/CCI'):
    !git clone https://github.com/IyadSultan/CCI.git /content/CCI
os.chdir('/content/CCI/session3/student')

!pip install -q openai pydantic pandas

import json
import pandas as pd
import sqlite3
from openai import OpenAI
from pydantic import BaseModel
from typing import List, Optional
from google.colab import userdata

# Make sure your API key is stored in Colab Secrets (click the key icon in the left sidebar)
# under the name: OPENAI_API_KEY
api_key = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

print(f"Working directory: {os.getcwd()}")
print("Setup complete!")

---
# Part 1: CSV to SQL — Load and Query Pharmacy Data (25%)

Load the VistA pharmacy export and patient demographics into a SQLite database,
explore the data, and write SQL queries to answer clinical pharmacy questions.

**Data files:**
- `../data/vista_pharmacy.csv` — 500 medication orders (122 columns)
- `../data/vista_patients.csv` — 79 patient demographics

**Key pharmacy columns:** `MRN`, `DRUG`, `QTY`, `DAYS_SUPPLY`, `Number_OF_REFILLS`, `ROUTE`, `SCHEDULE`, `STATUS`, `CLINIC`, `PROVIDER`, `PHARMACY_ORDERABLE_ITEM`, `DOSAGE_ORDERED`, `ISSUE_DATE`, `FILL_DATE`, `EXPIRATION_DATE`

**JOIN key:** `MRN` links pharmacy orders to patient demographics.

### Cell 1.1 — Load CSVs into SQLite

In [ ]:
# === CELL 1.1: LOAD DATA INTO SQLITE ===

# TODO: Load vista_pharmacy.csv into a DataFrame
# df_pharmacy = pd.read_csv('../data/vista_pharmacy.csv')

# TODO: Load vista_patients.csv into a DataFrame
# df_patients = pd.read_csv('../data/vista_patients.csv')

# TODO: Create a SQLite connection and load both tables
# conn = sqlite3.connect('khcc_pharmacy.db')
# df_pharmacy.to_sql('vista_pharmacy', conn, if_exists='replace', index=False)
# df_patients.to_sql('vista_patients', conn, if_exists='replace', index=False)

# TODO: Verify both tables loaded correctly
# Print row counts for each table


### Cell 1.2 — Explore Pharmacy Data

In [ ]:
# === CELL 1.2: EXPLORE PHARMACY DATA ===

# TODO: Print the shape of the pharmacy DataFrame

# TODO: Show the distribution of DRUG (top 10 most prescribed)

# TODO: Show the distribution of STATUS (ACTIVE, EXPIRED, DISCONTINUED, etc.)

# TODO: Show the distribution of ROUTE (ORAL, TOPICAL, SUBCUTANEOUS, etc.)

# TODO: How many unique patients have pharmacy orders?

# TODO: What is the date range of ISSUE_DATE?


### Cell 1.3 — SQL Queries (5 required)

Write these 5 SQL queries using `pd.read_sql()`:
1. **Top 10 prescribed drugs** — count of orders per drug, ordered descending
2. **Prescriptions per clinic** — count of orders per CLINIC
3. **Active vs expired** — count by STATUS
4. **Average days supply by drug** — for the top 5 drugs
5. **Patients with refills** — patients who have `Number_OF_REFILLS > 0`, show MRN, DRUG, refill count

In [ ]:
# === CELL 1.3: SQL QUERIES ===

# Query 1: Top 10 prescribed drugs
# TODO:

# Query 2: Prescriptions per clinic
# TODO:

# Query 3: Count by STATUS
# TODO:

# Query 4: Average days supply for top 5 drugs
# TODO:

# Query 5: Patients with refills > 0
# TODO:


### Cell 1.4 — JOIN Queries (3 required)

Combine pharmacy orders with patient demographics:
1. **Prescriptions by nationality** — JOIN on MRN, group by NATIONALITY
2. **Drugs for patients over 60** — calculate age from DOB, find their medications
3. **Male vs female prescription counts** — JOIN on MRN, group by SEX

In [ ]:
# === CELL 1.4: JOIN QUERIES ===

# JOIN Query 1: Prescriptions by nationality
# TODO:

# JOIN Query 2: Drugs for patients over 60 (calculate age from DOB)
# Hint: You can use julianday('now') - julianday(DOB) in SQLite
# TODO:

# JOIN Query 3: Male vs female prescription counts
# TODO:


---
# Part 2: Structured Output — Extract Drug Info from Clinical Notes (25%)

Use Pydantic structured output to extract medications, allergies, and potential drug interactions
from 10 synthetic oncology notes. Then cross-reference extracted medications against the
pharmacy database.

**Data file:** `../data/synthetic_clinical_notes.json`

### Cell 2.1 — Define Pydantic Models

In [ ]:
# === CELL 2.1: PYDANTIC MODELS ===

# TODO: Define a DrugMention model with:
#   name (str), dose (Optional[str]), frequency (Optional[str]), route (Optional[str])

# TODO: Define a DrugInteractionAlert model with:
#   drug1 (str), drug2 (str), risk_level (str: "high", "moderate", "low"),
#   description (str)

# TODO: Define a NoteExtraction model with:
#   patient_name (str), mrn (str), medications (List[DrugMention]),
#   allergies (List[str]), potential_interactions (List[DrugInteractionAlert]),
#   key_findings (List[str])


### Cell 2.2 — Load Clinical Notes

In [ ]:
# === CELL 2.2: LOAD CLINICAL NOTES ===

# TODO: Load the 10 notes from the data folder
# with open('../data/synthetic_clinical_notes.json', 'r') as f:
#     notes = json.load(f)

# TODO: Print how many notes loaded
# TODO: Print the first note's ID, type, and first 200 characters of text


### Cell 2.3 — Extract Structured Data from All 10 Notes

Use `client.beta.chat.completions.parse()` with your `NoteExtraction` Pydantic model
to extract structured medication and interaction data from each note.

In [ ]:
# === CELL 2.3: EXTRACT FROM ALL 10 NOTES ===

# TODO: Loop through all 10 notes
# For each note, call client.beta.chat.completions.parse() with:
#   model="gpt-4o-mini"
#   response_format=NoteExtraction
#   system prompt: instruct the model to extract medications, allergies,
#                  potential drug interactions, and key findings
#   user message: the note text

# TODO: Store all extractions in a list
# TODO: For each extraction, print:
#   - Note ID and patient name
#   - Number of medications found
#   - Number of potential interactions flagged
#   - Allergies

extractions = []
# for i, note_data in enumerate(notes):
#     ...


### Cell 2.4 — Cross-Reference with Pharmacy Database

For each medication extracted from the clinical notes, check if that drug
(by `PHARMACY_ORDERABLE_ITEM`) appears in the `vista_pharmacy` SQL table.
Report which extracted drugs are in the pharmacy system and which are not.

In [ ]:
# === CELL 2.4: CROSS-REFERENCE ===

# TODO: Get the list of unique PHARMACY_ORDERABLE_ITEM values from the database
# pharmacy_drugs = pd.read_sql("SELECT DISTINCT PHARMACY_ORDERABLE_ITEM FROM vista_pharmacy", conn)

# TODO: For each extraction, check if each medication name matches a pharmacy drug
# Use case-insensitive matching (e.g., compare .upper() versions)
# Print a summary: "TAMOXIFEN — FOUND in pharmacy" or "CISPLATIN — NOT FOUND"

# TODO: Report totals: X drugs matched, Y drugs not found


### Cell 2.5 — Compare Against Ground Truth

Each note has an `expected_extraction` field with ground truth for `medications`, `allergies`,
and `potential_interactions`. Compare your LLM extractions against these:
- How many medications did you correctly identify?
- Do the allergies match?
- How many drug-drug interaction pairs did the LLM find vs the ground truth?

In [ ]:
# === CELL 2.5: COMPARE AGAINST GROUND TRUTH ===

# TODO: For each note, compare:
#   - Number of medications extracted vs expected
#   - Whether allergies match
#   - Number of interaction pairs matched vs expected
#     Hint: normalize drug pairs with sorted() and .upper() for comparison
#   - Print a per-note accuracy summary

# TODO: Print overall summary:
#   - Total medications extracted vs total expected
#   - Total interactions extracted vs total expected
#   - Notes where extraction was complete vs partial


---
# Part 3: Text-to-SQL — Pharmacist Natural Language Interface (25%)

Build a system where a pharmacist can ask questions in plain English
and get answers from the pharmacy database — no SQL knowledge required.

### Cell 3.1 — Schema Description

Create a detailed schema description string for the LLM. Include:
- Both table names and key column names with types
- Sample values for important columns
- The JOIN relationship (MRN)
- Data quirks (many NULL columns, STATUS values, ROUTE values, date formats)

In [ ]:
# === CELL 3.1: SCHEMA DESCRIPTION ===

# TODO: Create a schema_description string that describes both tables
# Include the key pharmacy columns, patient columns, JOIN key, and data notes

schema_description = """
# TODO: Fill in the schema description
# Table: vista_pharmacy
# Columns: ...
#
# Table: vista_patients
# Columns: ...
#
# Relationship: ...
#
# Important notes: ...
"""

# TODO: Get sample rows from both tables to verify your schema
# pd.read_sql("SELECT DRUG, QTY, DAYS_SUPPLY, ROUTE, STATUS, CLINIC FROM vista_pharmacy LIMIT 3", conn)


### Cell 3.2 — Text-to-SQL Function

Write a function `text_to_sql(question)` that sends the schema + question
to GPT and gets back a SQL query. Include:
- Safety rules (SELECT only)
- Instruction to return only the SQL, no explanation

In [ ]:
# === CELL 3.2: TEXT TO SQL FUNCTION ===

# TODO: Write a function text_to_sql(question) that:
# 1. Builds a system prompt with the schema and safety rules
# 2. Sends the question to GPT-4o-mini
# 3. Returns the generated SQL string

def text_to_sql(question):
    pass  # TODO

# TODO: Test with: "How many total pharmacy orders are in the database?"


### Cell 3.3 — Full Pipeline with Few-Shot Examples

Build `ask_pharmacy(question)` that:
1. Generates SQL (using few-shot examples for better accuracy)
2. Validates it is a SELECT query
3. Executes it against the database
4. Sends results back to GPT for a natural language answer

Include at least **3 few-shot examples** in your system prompt, including one JOIN example.

In [ ]:
# === CELL 3.3: FULL PIPELINE WITH FEW-SHOT ===

# TODO: Write ask_pharmacy(question) that:
# 1. Calls text_to_sql() with few-shot examples
# 2. Validates it's a SELECT query (block dangerous keywords)
# 3. Executes the SQL with pd.read_sql()
# 4. Sends results to GPT for a natural language answer
# 5. Returns dict with: sql, answer, data

def ask_pharmacy(question):
    pass  # TODO


### Cell 3.4 — Test with 5 Pharmacy Questions

Run your pipeline on these 5 questions and print the SQL + answer for each:
1. "Which patients are on Tamoxifen?"
2. "What is the most prescribed drug in the ICU?"
3. "Show active prescriptions for patients from Amman"
4. "How many oral vs injectable medications are there?"
5. "Find patients with more than 5 pharmacy orders"

In [ ]:
# === CELL 3.4: TEST QUESTIONS ===

questions = [
    "Which patients are on Tamoxifen?",
    "What is the most prescribed drug in the ICU?",
    "Show active prescriptions for patients from Amman",
    "How many oral vs injectable medications are there?",
    "Find patients with more than 5 pharmacy orders"
]

# TODO: Loop through questions, call ask_pharmacy(), print SQL + answer
# for q in questions:
#     result = ask_pharmacy(q)
#     print(f"Q: {q}")
#     print(f"SQL: {result['sql']}")
#     print(f"Answer: {result['answer']}")
#     print()


---
# Part 4: Critical Analysis (15%)

Write **200-400 words** addressing all four questions below. This must reference YOUR specific implementation — generic answers will lose marks.

**Replace the TODOs below with your analysis.**

### 1. Drug Interaction Blind Spots
What can the LLM miss compared to a real clinical decision support system? Think about dose-dependent interactions, patient-specific contraindications (renal function, age), and temporal factors.

**TODO:** Your answer here (3-5 sentences)

### 2. Text-to-SQL Risks
What happens if the generated SQL is wrong but looks plausible? How would you validate results in a clinical pharmacy setting before acting on them?

**TODO:** Your answer here (3-5 sentences)

### 3. PHI / Security
If this pipeline were deployed at KHCC, what are the privacy and security risks? Consider: API key management, patient data sent to OpenAI, local regulations, and data residency.

**TODO:** Your answer here (3-5 sentences)

### 4. Missing Data
Many pharmacy columns are NULL in our synthetic data. Which missing columns would a real system need? (Think: NDC codes, allergy cross-checking, insurance/billing, drug-drug interaction databases.)

**TODO:** Your answer here (3-5 sentences)

---
# Part 5: Stretch Challenge (10%)

Choose **ONE** of the following options.

### Option A: Tool Calling Pharmacist Assistant

Define 2 tools:
- `run_sql_query(sql)` — executes a SELECT query and returns results
- `check_drug_interaction(drug1, drug2)` — returns interaction info

Build a multi-turn assistant that uses both tools to answer questions like:
"Is there an interaction between Tamoxifen and Anastrozole for patient MRN 123?"

### Option B: Text-to-SQL Benchmarking

Create **10 question / expected-SQL pairs**. Run each question through both:
- Zero-shot `text_to_sql()` (no examples)
- Few-shot `text_to_sql()` (with examples)

Compare accuracy: which questions improved with few-shot? Which did not? Report your findings.

In [ ]:
# === PART 5: STRETCH CHALLENGE ===
# TODO: Implement your chosen option (A or B)


---
## Submission Checklist

- [ ] Part 1: Pharmacy data loaded into SQLite with 5 SQL queries and 3 JOIN queries
- [ ] Part 2: Pydantic models defined, all 10 notes extracted, interactions compared to ground truth, cross-referenced with pharmacy DB
- [ ] Part 3: text_to_sql and ask_pharmacy functions working, 5 test questions answered
- [ ] Part 4: Critical analysis (200-400 words) referencing your implementation
- [ ] Part 5: Stretch challenge completed (Option A or B)
- [ ] Notebook runs top-to-bottom without errors
- [ ] Pushed to GitHub with 3+ descriptive commits
- [ ] No real patient data in the notebook